<a href="https://colab.research.google.com/github/LuciaMellini/AMD_project/blob/main/findingSimilarItems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding similar items

We download the Letterboxd dataset from Kaggle, using a token.

In [2]:
import os
import json
import pandas as pd
import pip
import string

os.environ['KAGGLE_USERNAME'] = "xxx"
os.environ['KAGGLE_KEY'] = "xxx"
! kaggle datasets download -d gsimonx37/letterboxd

Dataset URL: https://www.kaggle.com/datasets/gsimonx37/letterboxd
License(s): GPL-3.0
100% 22.5G/22.5G [04:43<00:00, 128MB/s]
100% 22.5G/22.5G [04:43<00:00, 85.3MB/s]


We only consider a subet of the files contained in the `letterboxd` dataset, namely the data regarding the movie names and ids, their actors, crews, genres and themes.

In [26]:
import zipfile
from multiprocessing import Pool

DATA_DIR = "./letterboxd"
members_to_extract = ['actors.csv', 'crew.csv', 'genres.csv', 'movies.csv', 'themes.csv']
with zipfile.ZipFile(DATA_DIR + ".zip","r") as zip_ref:
    for file_name in members_to_extract:
        zip_ref.extract(file_name, DATA_DIR)

!rm -rf {DATA_DIR + ".zip"}



We then prepare the entry point for the Spark functionalities that will we use from now on.

In [22]:
!apt-get install openjdk-21-jdk-headless -qq > /dev/null
!wget https://downloads.apache.org/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz
!tar xf spark-3.5.3-bin-hadoop3.tgz
!rm spark-3.5.3-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["SPARK_HOME"] = "spark-3.5.3-bin-hadoop3"

import findspark
findspark.init("spark-3.5.3-bin-hadoop3")
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

sc = spark.sparkContext

--2024-11-24 17:01:24--  https://downloads.apache.org/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz
Resolving downloads.apache.org (downloads.apache.org)... 135.181.214.104, 88.99.208.237, 2a01:4f9:3a:2c57::2, ...
Connecting to downloads.apache.org (downloads.apache.org)|135.181.214.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 400864419 (382M) [application/x-gzip]
Saving to: ‘spark-3.5.3-bin-hadoop3.tgz’

spark-3.5.3-bin-had 100%[===================>] 382.29M  19.3MB/s    in 22s     

2024-11-24 17:01:46 (17.7 MB/s) - ‘spark-3.5.3-bin-hadoop3.tgz’ saved [400864419/400864419]

